In [2]:
import pandas as pd
import duckdb as db
import requests
from io import StringIO

In [3]:
def get_data_from_API_call(url):
    """
    Send a GET request to the specified URL and return the response
    content as a file-like StringIO object.

    Parameters
    ----------
    url : str
        The complete endpoint URL from which data will be fetched.

    Returns
    -------
    io.StringIO
        A file-like object containing the UTF-8 decoded response text.

    Raises
    ------
    Exception
        If the HTTP status code is anything other than 200, an
        exception is raised with a message that includes the received
        status code.
    """
    # Send HTTP GET request
    response = requests.get(url)
    # Check if the request was successful
    if response.status_code == 200:
        data = StringIO(response.content.decode("utf-8"))
        return pd.read_csv(data)
    else:
        print("Failed to retrieve data. Status code:", response.status_code)    
        raise Exception(f"Failed to retrieve data. Status code: {response.status_code}")

# TrafficPerTerritory

In [4]:
# url passenger data
url_pas = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000013/~latest.csv?lang=en&representation=TIME_PERIOD[~last=1]&granularity=TIME_PERIOD[M]"
# url goods and mail data
url_gm = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000014/~latest.csv?lang=en&representation=TIME_PERIOD[~last=1]&granularity=TIME_PERIOD[M]"
# url operations
url_o = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000015/~latest.csv?lang=en&representation=TIME_PERIOD[~last=1]&granularity=TIME_PERIOD[M]"

In [5]:
pas = get_data_from_API_call(url_pas)
gm = get_data_from_API_call(url_gm)
op = get_data_from_API_call(url_o)

In [6]:
dfs = [pas, gm, op]

In [7]:
tpt = db.read_csv("../../data/TrafficPerTerritory.csv")

In [10]:
""" 
The standard DuckDB Python API provides a SQL interface compliant with the DB-API 2.0 
specification described by PEP 249 similar to the SQLite Python API
https://peps.python.org/pep-0249/
"""
max_date_local = db.sql(' \
SELECT MAX(Month) FROM tpt \
').fetchone()[0]

max_date_local

datetime.date(2025, 7, 1)

In [23]:
for df in dfs:
    df['Month'] = pd.to_datetime(df['TIME_PERIOD#en'], format="%m/%Y").dt.date
    max_date_df = db.sql(' \
        SELECT MAX(Month) FROM df \
        ').fetchone()[0]
    if max_date_df > max_date_local:
        print("There is new data")
        # If the difference is more than one month, then you skipped a month, shouldnt happen but we make the error just in case
    elif max_date_df == max_date_local:
        print("There is no new data")
    else:
        print("The stored table has data ")

There is no new data
There is no new data
There is no new data


# TrafficPerAirport